In [3]:
#installing dependencies

%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
# creating an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

In [6]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    # Send request
    resp = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )

    # Normalize possible response shapes and extract text pieces
    content = None
    if hasattr(resp, 'content'):
        content = resp.content
    elif isinstance(resp, dict):
        content = resp.get('content')

    text_parts = []
    if isinstance(content, list):
        for item in content:
            # item may be a plain string
            if isinstance(item, str):
                text_parts.append(item)
                continue

            # try attribute access
            if hasattr(item, 'text'):
                try:
                    text_parts.append(getattr(item, 'text'))
                    continue
                except Exception:
                    pass

            # try dict-style access
            if isinstance(item, dict) and 'text' in item:
                text_parts.append(item['text'])
                continue

            # last resort: stringify the item
            text_parts.append(str(item))

    else:
        # Fallbacks if `content` isn't a list
        if hasattr(resp, 'text'):
            return getattr(resp, 'text')
        if isinstance(resp, dict) and 'text' in resp:
            return resp['text']
        # As a final fallback, return the string representation
        return str(resp)

    return ''.join(text_parts)

In [ ]:
messages = []

while True:

    # Get user input 
    user_input = input("> ")
    print(">", user_input)

    add_user_message(messages, user_input)
    answer = chat(messages)
    add_assistant_message(messages, answer)
    print(answer)

> whats 1 + 1
> whats the answer x 2
> 


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.4: user messages must have non-empty content'}, 'request_id': 'req_011CdAep35MEUeZSKwp6v6Yz'}

In [ ]:
# # Make a starting list of messages
# messages = []

# # Add in the initial user question of "Define quantum computing in one sentence"
# add_user_message(messages, "Define quantum computing in one sentence")

# # Pass the list of messages into 'chat' to get an answer
# answer = chat(messages)

# # Take the answer and add it as an assistant message into our list
# add_assistant_message(messages, answer)

# # Add in the user's follow-up question
# add_user_message(messages, "Write another sentence")

# # Call chat again with the list of messages to get a final answer
# answer = chat(messages)
# answer



'Unlike classical bits, which represent either a 0 or a 1, quantum bits (qubits) can exist in multiple states simultaneously, enabling quantum computers to explore many possible solutions in parallel.'